In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
from datasets import load_dataset

load_dotenv()

hf_token = os.getenv("HF_TOKEN")
name = "minicpm-v_8b_discrete"
ds = load_dataset(f"Emotion-Aware-AI-Assistant/{name}_standardized", token=hf_token)
df = pd.DataFrame(ds['train'])

c:\Users\FernandaBufon\miniconda3\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\FernandaBufon\miniconda3\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\FernandaBufon\.cache\huggingface\hub\datasets--Emotion-Aware-AI-Assistant--minicpm-v_8b_discrete_standardized. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode

In [3]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
import pandas as pd
import numpy as np
import json

# -----------------------------------------------------------
# Definir classes válidas e inválidas
# -----------------------------------------------------------
EMOTIONS = ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']
INVALID = ['wrong_format', 'emotion_refused', 'emotion_not_listed']
ALL_CLASSES = EMOTIONS + INVALID

# -----------------------------------------------------------
# Filtrar somente labels válidas
# -----------------------------------------------------------
df_eval = df[df["label"].isin(EMOTIONS)]

y_true = df_eval["label"].tolist()
y_pred = df_eval["predicted_emotion"].tolist()

# -----------------------------------------------------------
# MÉTRICAS MACRO
# -----------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", labels=EMOTIONS)
macro_precision = precision_score(y_true, y_pred, average="macro", labels=EMOTIONS)
macro_recall = recall_score(y_true, y_pred, average="macro", labels=EMOTIONS)

# -----------------------------------------------------------
# MÉTRICAS POR CLASSE
# -----------------------------------------------------------
f1_per_class = f1_score(y_true, y_pred, average=None, labels=EMOTIONS)
precision_per_class = precision_score(y_true, y_pred, average=None, labels=EMOTIONS)
recall_per_class = recall_score(y_true, y_pred, average=None, labels=EMOTIONS)

# -----------------------------------------------------------
# DF_RESULTS (MÉTRICAS)
# -----------------------------------------------------------
data = {
    "accuracy": [accuracy],
    "macro_f1": [macro_f1],
    "macro_precision": [macro_precision],
    "macro_recall": [macro_recall],
}

for emotion, value in zip(EMOTIONS, f1_per_class):
    data[f"f1_{emotion}"] = [value]

for emotion, value in zip(EMOTIONS, precision_per_class):
    data[f"precision_{emotion}"] = [value]

for emotion, value in zip(EMOTIONS, recall_per_class):
    data[f"recall_{emotion}"] = [value]

df_results = pd.DataFrame(data)

# -----------------------------------------------------------
# MATRIZ DE CONFUSÃO EXPANDIDA
# -----------------------------------------------------------

# Garantir que previsões fora da lista sejam preservadas
y_pred_expanded = [
    p if p in ALL_CLASSES else "other_invalid"
    for p in y_pred
]

if "other_invalid" in y_pred_expanded:
    ALL_CLASSES.append("other_invalid")

# Confusion matrix numérica
cm = confusion_matrix(
    y_true,
    y_pred_expanded,
    labels=ALL_CLASSES
)

df_confusion = pd.DataFrame(cm, index=ALL_CLASSES, columns=ALL_CLASSES)

# -----------------------------------------------------------
# ADICIONAR MATRIZ COMO JSON DENTRO DE df_results
# -----------------------------------------------------------
confusion_json = df_confusion.to_dict()     # vira um dict {row: {col: value}}
confusion_json_str = json.dumps(confusion_json)

df_results["confusion_matrix_json"] = confusion_json_str

# Mostrar resultado
df_results


,accuracy,macro_f1,macro_precision,macro_recall,f1_anger,f1_disgust,f1_fear,f1_happiness,f1_neutral,f1_sadness,...,precision_sadness,precision_surprise,recall_anger,recall_disgust,recall_fear,recall_happiness,recall_neutral,recall_sadness,recall_surprise,confusion_matrix_json
0,0.303084,0.29754,0.426439,0.263177,0.374363,0.036847,0.095522,0.478472,0.368968,0.350313,...,0.655518,0.485024,0.508651,0.020525,0.090141,0.3302,0.343633,0.239024,0.310068,"{""anger"": {""anger"": 441, ""disgust"": 220, ""fear..."


In [7]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix
)
import pandas as pd
import json

# -----------------------------------------------------------
# Definir classes válidas
# -----------------------------------------------------------
EMOTIONS = ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']

INVALID_PREDICTIONS = ["wrong_format", "emotion_refused", "emotion_not_listed"]

# -----------------------------------------------------------
# Filtrar somente labels válidos
# -----------------------------------------------------------
df_eval = df[df["label"].isin(EMOTIONS)]

y_true = df_eval["label"]
y_pred = df_eval["predicted_emotion"]

# -----------------------------------------------------------
# Corrigir previsões inválidas para "INVALID"
# -----------------------------------------------------------
y_pred_clean = [
    pred if pred in EMOTIONS else "INVALID"
    for pred in y_pred
]

# -----------------------------------------------------------
# Métricas macro
# -----------------------------------------------------------
accuracy = accuracy_score(y_true, y_pred_clean)
macro_f1 = f1_score(y_true, y_pred_clean, average="macro", labels=EMOTIONS)
macro_precision = precision_score(y_true, y_pred_clean, average="macro", labels=EMOTIONS)
macro_recall = recall_score(y_true, y_pred_clean, average="macro", labels=EMOTIONS)

# -----------------------------------------------------------
# Métricas por classe
# -----------------------------------------------------------
f1_per_class = f1_score(y_true, y_pred_clean, average=None, labels=EMOTIONS)
precision_per_class = precision_score(y_true, y_pred_clean, average=None, labels=EMOTIONS)
recall_per_class = recall_score(y_true, y_pred_clean, average=None, labels=EMOTIONS)

# -----------------------------------------------------------
# Matriz de confusão como JSON
# -----------------------------------------------------------
labels_with_invalid = EMOTIONS + ["INVALID"]
cm = confusion_matrix(y_true, y_pred_clean, labels=labels_with_invalid)

df_confusion = pd.DataFrame(cm, index=labels_with_invalid, columns=labels_with_invalid)
confusion_json = df_confusion.to_json()

# -----------------------------------------------------------
# Erros por motivo inválido por emoção (df_invalid)
# -----------------------------------------------------------
invalid_counts = {}

for emotion in EMOTIONS:
    subset = df_eval[df_eval["label"] == emotion]
    invalid_counts[emotion] = {
        reason: int((subset["predicted_emotion"] == reason).sum())
        for reason in INVALID_PREDICTIONS
    }

df_invalid = pd.DataFrame.from_dict(invalid_counts, orient="index")
df_invalid.columns = INVALID_PREDICTIONS

invalid_json = df_invalid.to_json()

# -----------------------------------------------------------
# Construir df_results com JSON embutido
# -----------------------------------------------------------
data = {
    "accuracy": [accuracy],
    "macro_f1": [macro_f1],
    "macro_precision": [macro_precision],
    "macro_recall": [macro_recall],
    "confusion_matrix_json": [confusion_json],
    "invalid_counts_json": [invalid_json],
}

# Adicionar F1 por classe
for emotion, value in zip(EMOTIONS, f1_per_class):
    data[f"f1_{emotion}"] = [value]

# Adicionar Precision por classe
for emotion, value in zip(EMOTIONS, precision_per_class):
    data[f"precision_{emotion}"] = [value]

# Adicionar Recall por classe
for emotion, value in zip(EMOTIONS, recall_per_class):
    data[f"recall_{emotion}"] = [value]

df_results = pd.DataFrame(data)

# Mostrar resultado final
df_results


,accuracy,macro_f1,macro_precision,macro_recall,confusion_matrix_json,invalid_counts_json,f1_anger,f1_disgust,f1_fear,f1_happiness,...,precision_neutral,precision_sadness,precision_surprise,recall_anger,recall_disgust,recall_fear,recall_happiness,recall_neutral,recall_sadness,recall_surprise
0,0.303084,0.29754,0.426439,0.263177,"{""anger"":{""anger"":441,""disgust"":220,""fear"":61,...","{""wrong_format"":{""anger"":163,""disgust"":176,""fe...",0.374363,0.036847,0.095522,0.478472,...,0.398336,0.655518,0.485024,0.508651,0.020525,0.090141,0.3302,0.343633,0.239024,0.310068
